In [137]:
import random
import time

import tensorflow as tf
from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix,  precision_score, recall_score, f1_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler,  LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd


In [138]:
column_names = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty",
]



train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Train.csv",names=column_names, header=None)
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Test.csv",names=column_names, header=None)

In [139]:
test_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty'],
      dtype='object')

In [140]:
test_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,NaN
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,NaN
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,NaN
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,NaN
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,NaN


In [141]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)
print(train_df.head())

Training data shape: (125973, 43)
Testing data shape: (22544, 43)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1     

In [142]:
test_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty'],
      dtype='object')

In [88]:
test_df['label'].value_counts()

,count
label,
normal,9711
neptune,4657
guess_passwd,1231
mscan,996
warezmaster,944
apache2,737
satan,735
processtable,685
smurf,665


In [143]:
all_df = pd.concat([train_df, test_df], axis=0)

In [148]:
train_labels = (
    all_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
)

test_labels = (
    test_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
)

y_train_all = (train_labels != "normal").astype(np.int32).to_numpy()
y_test = (test_labels != "normal").astype(np.int32).to_numpy()

print("Training label values:", np.unique(y_train_all))
print("Testing label values:", np.unique(y_test))

Training label values: [0 1]
Testing label values: [0 1]


In [149]:
X_train_all_df = all_df.drop(
    columns=["label", "difficulty"]
).copy()

X_test_df = test_df.drop(
    columns=["label", "difficulty"]
).copy()

print("Training features:", X_train_all_df.shape)
print("Testing features: ", X_test_df.shape)

Training features: (148517, 41)
Testing features:  (22544, 41)


# **An Empirical Study on Network Anomaly Detection using Convolutional Neural Networks [33]**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
categorical_columns = [
    "protocol_type",
    "service",
    "flag",
]

numerical_columns = [
    column
    for column in X_train_df.columns
    if column not in categorical_columns
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_columns,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_columns,
        ),
    ]
)

In [ ]:
X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

print("Processed training shape:", X_train.shape)
print("Processed testing shape:", X_test.shape)

Processed training shape: (125973, 122)
Processed testing shape: (22544, 122)


In [ ]:
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print("CNN training shape:", X_train.shape)
print("CNN testing shape:", X_test.shape)

CNN training shape: (125973, 122, 1)
CNN testing shape: (22544, 122, 1)


In [ ]:
y_train_encoded = to_categorical(y_train, num_classes=2)
y_test_encoded = to_categorical(y_test, num_classes=2)

print(y_train_encoded.shape)
print(y_train_encoded[:5])

(125973, 2)
[[1. 0.]
 [1. 0.]
 [0. 1.]
 [1. 0.]
 [1. 0.]]


In [ ]:
model = Sequential([
    Input(shape=(122, 1)),

    Conv1D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
    ),
    Activation("relu"),
    BatchNormalization(),
    Dropout(rate=0.5),

    MaxPooling1D(
        pool_size=2,
        strides=2,
        padding="valid",
    ),

    Flatten(),

    Dense(
        units=64,
        use_bias=False,
    ),
    Activation("relu"),
    BatchNormalization(),
    Dropout(rate=0.5),

    Dense(
        units=2,
        activation="softmax",
    ),
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 122, 64)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 122, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 122, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 122, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 61, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3904)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       249,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 250,690 (979.26 KB)

 Trainable params: 250,434 (978.26 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
y_train_encoded = to_categorical(y_train, num_classes=2)
y_test_encoded = to_categorical(y_test, num_classes=2)

In [ ]:
history = model.fit(
    X_train,
    y_train_encoded,
    epochs=10,
    batch_size=32,
    shuffle=True,
    verbose=2,
)

Epoch 1/10
3937/3937 - 17s - 4ms/step - accuracy: 0.9761 - loss: 0.0696
Epoch 2/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9819 - loss: 0.0510
Epoch 3/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9838 - loss: 0.0457
Epoch 4/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9849 - loss: 0.0433
Epoch 5/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9849 - loss: 0.0428
Epoch 6/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9860 - loss: 0.0404
Epoch 7/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9864 - loss: 0.0399
Epoch 8/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9856 - loss: 0.0407
Epoch 9/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9868 - loss: 0.0386
Epoch 10/10
3937/3937 - 7s - 2ms/step - accuracy: 0.9867 - loss: 0.0375


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test_encoded,
    batch_size=32,
    verbose=0,
)

print("\nTest results")
print("------------")
print(f"Loss:     {test_loss:.6f}")
print(f"Accuracy: {test_accuracy:.6f}")
print(f"Accuracy: {test_accuracy * 100:.2f}%")


Test results
------------
Loss:     1.431539
Accuracy: 0.760158
Accuracy: 76.02%


In [ ]:

predicted_probabilities = model.predict(
    X_test,
    batch_size=32,
    verbose=0,
)

y_pred = np.argmax(
    predicted_probabilities,
    axis=1,
)


cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()


print("\nClassification report")
print("---------------------")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["normal", "attack"],
        digits=4,
        zero_division=0,
    )
)

attack_precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    average="binary",
    zero_division=0,
)

attack_detection_rate = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    average="binary",
    zero_division=0,
)

false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

attack_f1_score = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    average="binary",
    zero_division=0,
)




print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Detection rate : {attack_detection_rate:.4f} "
      f"({attack_detection_rate * 100:.2f}%)")
print(f"Precision      : {attack_precision:.4f} "
      f"({attack_precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {attack_f1_score:.4f} "
      f"({attack_f1_score * 100:.2f}%)")





Classification report
---------------------
              precision    recall  f1-score   support

      normal     0.6512    0.9546    0.7742      9711
      attack     0.9469    0.6130    0.7442     12833

    accuracy                         0.7602     22544
   macro avg     0.7990    0.7838    0.7592     22544
weighted avg     0.8195    0.7602    0.7572     22544

Confusion matrix
----------------
[[9270  441]
 [4966 7867]]

Attack-class metrics
--------------------
Detection rate : 0.6130 (61.30%)
Precision      : 0.9469 (94.69%)
FPR            : 0.0454 (4.54%)
F1 score       : 0.7442 (74.42%)


In [ ]:
model.save("/content/drive/MyDrive/1DCNN_Kwon.keras")

# **Deep Learning Approach for Intelligent Intrusion Detection System [69]**

In [150]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [152]:
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X_train_all_df,
    y_train_all,
    test_size=0.05,
    random_state=SEED,
    stratify=y_train_all,
)

print("\nData split")
print("----------")
print("Training:   ", X_train_df.shape)
print("Validation: ", X_val_df.shape)
print("Testing:    ", X_test_df.shape)


Data split
----------
Training:    (141091, 41)
Validation:  (7426, 41)
Testing:     (22544, 41)


In [153]:
categorical_columns = [
    "protocol_type",
    "service",
    "flag",
]

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

X_train_df.loc[:, categorical_columns] = encoder.fit_transform(
    X_train_df[categorical_columns]
)

X_val_df.loc[:, categorical_columns] = encoder.transform(
    X_val_df[categorical_columns]
)

X_test_df.loc[:, categorical_columns] = encoder.transform(
    X_test_df[categorical_columns]
)



X_train = X_train_df.astype(np.float32).to_numpy()
X_val = X_val_df.astype(np.float32).to_numpy()
X_test = X_test_df.astype(np.float32).to_numpy()

In [154]:
normalizer = Normalizer(norm="l2")

X_train = normalizer.fit_transform(X_train).astype(np.float32)
X_val = normalizer.transform(X_val).astype(np.float32)
X_test = normalizer.transform(X_test).astype(np.float32)

print("\nProcessed shapes")
print("----------------")
print("Training:   ", X_train.shape)
print("Validation: ", X_val.shape)
print("Testing:    ", X_test.shape)


Processed shapes
----------------
Training:    (141091, 41)
Validation:  (7426, 41)
Testing:     (22544, 41)


In [155]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(41,),
            name="nsl_kdd_input",
        ),

        tf.keras.layers.Dense(
            units=1024,
            activation="relu",
            name="dense_1024",
        ),
        tf.keras.layers.BatchNormalization(
            name="batch_norm_1024",
        ),
        tf.keras.layers.Dropout(
            rate=0.01,
            name="dropout_1024",
        ),

        tf.keras.layers.Dense(
            units=768,
            activation="relu",
            name="dense_768",
        ),
        tf.keras.layers.BatchNormalization(
            name="batch_norm_768",
        ),
        tf.keras.layers.Dropout(
            rate=0.01,
            name="dropout_768",
        ),

        tf.keras.layers.Dense(
            units=512,
            activation="relu",
            name="dense_512",
        ),
        tf.keras.layers.BatchNormalization(
            name="batch_norm_512",
        ),
        tf.keras.layers.Dropout(
            rate=0.01,
            name="dropout_512",
        ),

        tf.keras.layers.Dense(
            units=256,
            activation="relu",
            name="dense_256",
        ),
        tf.keras.layers.BatchNormalization(
            name="batch_norm_256",
        ),
        tf.keras.layers.Dropout(
            rate=0.01,
            name="dropout_256",
        ),

        tf.keras.layers.Dense(
            units=128,
            activation="relu",
            name="dense_128",
        ),
        tf.keras.layers.BatchNormalization(
            name="batch_norm_128",
        ),
        tf.keras.layers.Dropout(
            rate=0.01,
            name="dropout_128",
        ),
        tf.keras.layers.Dense(
            units=1,
            activation="sigmoid",
            name="binary_output",
        ),
    ],
    name="nsl_kdd_binary_dnn",
)

In [156]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.1,
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5,
        ),
        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5,
        ),
        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5,
        ),
        tf.keras.metrics.AUC(
            name="auc",
        ),
    ],
)

model.summary()

Model: "nsl_kdd_binary_dnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1024 (Dense)              │ (None, 1024)           │        43,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1024                 │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1024 (Dropout)          │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_768 (Dense)               │ (None, 768)            │       787,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_768                  │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_768 (Dropout)           │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_512 (Dense)               │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_512                  │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_512 (Dropout)           │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_256 (Dense)               │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_256                  │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_256 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_128                  │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_128 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ binary_output (Dense)           │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,399,041 (5.34 MB)

 Trainable params: 1,393,665 (5.32 MB)

 Non-trainable params: 5,376 (21.00 KB)

In [157]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=64,
    shuffle=True,
    verbose=2,
)


Epoch 1/200
2205/2205 - 36s - 16ms/step - accuracy: 0.9240 - auc: 0.9802 - loss: 0.1806 - precision: 0.9307 - recall: 0.9097 - val_accuracy: 0.9271 - val_auc: 0.9881 - val_loss: 0.1398 - val_precision: 0.9100 - val_recall: 0.9418
Epoch 2/200
2205/2205 - 6s - 3ms/step - accuracy: 0.9358 - auc: 0.9844 - loss: 0.1582 - precision: 0.9426 - recall: 0.9229 - val_accuracy: 0.9175 - val_auc: 0.9866 - val_loss: 12592.6396 - val_precision: 0.8615 - val_recall: 0.9871
Epoch 3/200
2205/2205 - 6s - 3ms/step - accuracy: 0.9484 - auc: 0.9892 - loss: 0.1291 - precision: 0.9523 - recall: 0.9398 - val_accuracy: 0.9414 - val_auc: 0.9807 - val_loss: 11961312.0000 - val_precision: 0.9280 - val_recall: 0.9521
Epoch 4/200
2205/2205 - 6s - 3ms/step - accuracy: 0.9533 - auc: 0.9906 - loss: 0.1193 - precision: 0.9579 - recall: 0.9444 - val_accuracy: 0.9379 - val_auc: 0.9763 - val_loss: 4876812800.0000 - val_precision: 0.9924 - val_recall: 0.8777
Epoch 5/200
2205/2205 - 6s - 3ms/step - accuracy: 0.9583 - auc: 0.

In [158]:
y_probability = model.predict(
    X_test,
    verbose=0,
).reshape(-1)

y_pred = (y_probability >= 0.5).astype(np.int32)
y_test_int = y_test.astype(np.int32)

In [159]:
cm = confusion_matrix(y_test_int, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_int, y_pred)

recall = recall_score(
    y_test_int,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_int,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_int,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[ 9461   250]
 [  342 12491]]

Attack-class metrics
--------------------
Accuracy : 0.9737 (97.37%)
Detection rate : 0.9733 (97.33%)
Precision      : 0.9804 (98.04%)
FPR            : 0.0257 (2.57%)
F1 score       : 0.9769 (97.69%)


In [ ]:
model.save("/content/drive/MyDrive/Models_Table_5/DNN_Vinayakumar.keras")

# **Attack classification of an intrusion detection system using deep learning and hyperparameter optimization [31]**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

VALIDATION_SIZE = 0.20
BATCH_SIZE = 256
LEARNING_RATE = 0.01
DAE_EPOCHS = 100
DNN_EPOCHS = 200

In [ ]:
NUMBER_OF_FEATURES = 122
NUMBER_OF_CLASSES = 5

CLASS_NAMES = [
    "Normal",
    "DoS",
    "Probe",
    "R2L",
    "U2R",
]

FEATURE_COLUMNS = column_names[:41]
CATEGORICAL_COLUMNS = [
    "protocol_type",
    "service",
    "flag",
]

NUMERICAL_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column not in CATEGORICAL_COLUMNS
]

In [ ]:
ATTACK_GROUPS = {
    "Normal": {
        "normal",
    },

    "DoS": {
        "apache2",
        "back",
        "mailbomb",
        "processtable",
        "teardrop",
        "smurf",
        "land",
        "neptune",
        "pod",
        "udpstorm",
    },

    "Probe": {
        "nmap",
        "ipsweep",
        "portsweep",
        "satan",
        "mscan",
        "saint",
    },

    "R2L": {
        "ftp_write",
        "guess_passwd",
        "snmpgetattack",
        "snmpguess",
        "imap",
        "spy",
        "warezclient",
        "warezmaster",
        "multihop",
        "phf",
        "named",
        "sendmail",
        "xlock",
        "xsnoop",
        "worm",
    },

    "U2R": {
        "ps",
        "buffer_overflow",
        "perl",
        "rootkit",
        "loadmodule",
        "xterm",
        "sqlattack",
        "httptunnel",
    },
}


ATTACK_TO_CLASS = {
    attack_name: class_name
    for class_name, attack_names in ATTACK_GROUPS.items()
    for attack_name in attack_names
}

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

In [ ]:
train_attack_names = (
    train_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.lower()
)

test_attack_names = (
    test_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.lower()
)

In [ ]:
train_class_names = train_attack_names.map(ATTACK_TO_CLASS)
test_class_names = test_attack_names.map(ATTACK_TO_CLASS)

y_train_all = (
    train_class_names
    .map(CLASS_TO_INDEX)
    .to_numpy(dtype=np.int32)
)

y_test = (
    test_class_names
    .map(CLASS_TO_INDEX)
    .to_numpy(dtype=np.int32)
)

In [ ]:
X_train_df = train_df[FEATURE_COLUMNS].copy()
X_test_df = test_df[FEATURE_COLUMNS].copy()

In [ ]:
categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

X_train_categorical = categorical_encoder.fit_transform(
    X_train_df[CATEGORICAL_COLUMNS]
)

X_test_categorical = categorical_encoder.transform(
    X_test_df[CATEGORICAL_COLUMNS]
)



X_train_numerical = (
    X_train_df[NUMERICAL_COLUMNS]
    .astype(np.float32)
    .to_numpy()
)

X_test_numerical = (
    X_test_df[NUMERICAL_COLUMNS]
    .astype(np.float32)
    .to_numpy()
)



X_train_all = np.concatenate(
    [
        X_train_numerical,
        X_train_categorical,
    ],
    axis=1,
)

X_test = np.concatenate(
    [
        X_test_numerical,
        X_test_categorical,
    ],
    axis=1,
).astype(np.float32)

In [ ]:
scaler = MinMaxScaler(
        feature_range=(0.0, 1.0),
        clip=True,
    )

X_train_all = scaler.fit_transform(
    X_train_all
).astype(np.float32)

X_test = scaler.transform(
    X_test
).astype(np.float32)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all,
    y_train_all,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=y_train_all,
)


# One-hot encode the five output classes

y_train_encoded = tf.keras.utils.to_categorical(
    y_train,
    num_classes=NUMBER_OF_CLASSES,
).astype(np.float32)

y_val_encoded = tf.keras.utils.to_categorical(
    y_val,
    num_classes=NUMBER_OF_CLASSES,
).astype(np.float32)

y_test_encoded = tf.keras.utils.to_categorical(
    y_test,
    num_classes=NUMBER_OF_CLASSES,
).astype(np.float32)

In [ ]:
autoencoder_input = tf.keras.layers.Input(
    shape=(NUMBER_OF_FEATURES,),
    name="autoencoder_input",
)

encoder_100 = tf.keras.layers.Dense(
    units=100,
    activation="relu",
    kernel_initializer="he_uniform",
    name="encoder_100",
)(autoencoder_input)

bottleneck_20 = tf.keras.layers.Dense(
    units=20,
    activation="relu",
    kernel_initializer="he_uniform",
    name="bottleneck_20",
)(encoder_100)

decoder_100 = tf.keras.layers.Dense(
    units=100,
    activation="relu",
    kernel_initializer="he_uniform",
    name="decoder_100",
)(bottleneck_20)

reconstruction_output = tf.keras.layers.Dense(
    units=NUMBER_OF_FEATURES,
    activation="sigmoid",
    kernel_initializer="he_uniform",
    name="reconstruction_output",
)(decoder_100)

autoencoder = tf.keras.Model(
    inputs=autoencoder_input,
    outputs=reconstruction_output,
    name="nsl_kdd_deep_autoencoder",
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
)

print("\nDeep autoencoder")
print("----------------")
autoencoder.summary()


Deep autoencoder
----------------


Model: "nsl_kdd_deep_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ autoencoder_input (InputLayer)  │ (None, 122)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_100 (Dense)             │ (None, 100)            │        12,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck_20 (Dense)           │ (None, 20)             │         2,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_100 (Dense)             │ (None, 100)            │         2,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction_output (Dense)   │ (None, 122)            │        12,322 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,742 (112.27 KB)

 Trainable params: 28,742 (112.27 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
dae_history = autoencoder.fit(
    X_train,
    X_train,
    validation_data=(X_val, X_val),
    epochs=DAE_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2,
)

Epoch 1/100
394/394 - 37s - 93ms/step - loss: 0.0347 - val_loss: 0.0160
Epoch 2/100
394/394 - 1s - 2ms/step - loss: 0.0154 - val_loss: 0.0151
Epoch 3/100
394/394 - 1s - 2ms/step - loss: 0.0148 - val_loss: 0.0147
Epoch 4/100
394/394 - 1s - 2ms/step - loss: 0.0145 - val_loss: 0.0145
Epoch 5/100
394/394 - 1s - 2ms/step - loss: 0.0144 - val_loss: 0.0144
Epoch 6/100
394/394 - 1s - 2ms/step - loss: 0.0143 - val_loss: 0.0144
Epoch 7/100
394/394 - 1s - 2ms/step - loss: 0.0142 - val_loss: 0.0144
Epoch 8/100
394/394 - 1s - 2ms/step - loss: 0.0142 - val_loss: 0.0142
Epoch 9/100
394/394 - 1s - 2ms/step - loss: 0.0141 - val_loss: 0.0142
Epoch 10/100
394/394 - 1s - 2ms/step - loss: 0.0141 - val_loss: 0.0141
Epoch 11/100
394/394 - 1s - 2ms/step - loss: 0.0141 - val_loss: 0.0142
Epoch 12/100
394/394 - 1s - 2ms/step - loss: 0.0140 - val_loss: 0.0142
Epoch 13/100
394/394 - 1s - 2ms/step - loss: 0.0140 - val_loss: 0.0140
Epoch 14/100
394/394 - 1s - 2ms/step - loss: 0.0140 - val_loss: 0.0141
Epoch 15/100


In [ ]:
classifier = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(NUMBER_OF_FEATURES,),
            name="classifier_input",
        ),

        tf.keras.layers.Dense(
            units=100,
            activation="relu",
            kernel_initializer="he_uniform",
            name="classifier_encoder_100",
        ),

        tf.keras.layers.Dense(
            units=20,
            activation="relu",
            kernel_initializer="he_uniform",
            name="classifier_bottleneck_20",
        ),

        tf.keras.layers.Dense(
            units=NUMBER_OF_CLASSES,
            activation="softmax",
            kernel_initializer="he_uniform",
            name="multiclass_output",
        ),
    ],
    name="nsl_kdd_ptdae_dnn",
)



_ = classifier(
    np.zeros(
        (1, NUMBER_OF_FEATURES),
        dtype=np.float32,
    )
)


In [ ]:
classifier.get_layer(
    "classifier_encoder_100"
).set_weights(
    autoencoder.get_layer(
        "encoder_100"
    ).get_weights()
)

classifier.get_layer(
    "classifier_bottleneck_20"
).set_weights(
    autoencoder.get_layer(
        "bottleneck_20"
    ).get_weights()
)



classifier.get_layer(
    "classifier_encoder_100"
).trainable = True

classifier.get_layer(
    "classifier_bottleneck_20"
).trainable = True

In [ ]:
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(
            name="accuracy",
        ),
    ],
)

classifier.summary()

Model: "nsl_kdd_ptdae_dnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ classifier_encoder_100 (Dense)  │ (None, 100)            │        12,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier_bottleneck_20        │ (None, 20)             │         2,020 │
│ (Dense)                         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiclass_output (Dense)       │ (None, 5)              │           105 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,425 (56.35 KB)

 Trainable params: 14,425 (56.35 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
dnn_history = classifier.fit(
    X_train,
    y_train_encoded,
    validation_data=(
        X_val,
        y_val_encoded,
    ),
    epochs=DNN_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2,
)

Epoch 1/200
394/394 - 9s - 22ms/step - accuracy: 0.9684 - loss: 0.1041 - val_accuracy: 0.9869 - val_loss: 0.0351
Epoch 2/200
394/394 - 1s - 2ms/step - accuracy: 0.9915 - loss: 0.0256 - val_accuracy: 0.9899 - val_loss: 0.0259
Epoch 3/200
394/394 - 1s - 2ms/step - accuracy: 0.9934 - loss: 0.0197 - val_accuracy: 0.9900 - val_loss: 0.0242
Epoch 4/200
394/394 - 1s - 2ms/step - accuracy: 0.9941 - loss: 0.0171 - val_accuracy: 0.9902 - val_loss: 0.0242
Epoch 5/200
394/394 - 1s - 2ms/step - accuracy: 0.9945 - loss: 0.0158 - val_accuracy: 0.9906 - val_loss: 0.0241
Epoch 6/200
394/394 - 1s - 2ms/step - accuracy: 0.9950 - loss: 0.0148 - val_accuracy: 0.9923 - val_loss: 0.0207
Epoch 7/200
394/394 - 1s - 2ms/step - accuracy: 0.9954 - loss: 0.0136 - val_accuracy: 0.9933 - val_loss: 0.0199
Epoch 8/200
394/394 - 1s - 2ms/step - accuracy: 0.9955 - loss: 0.0134 - val_accuracy: 0.9919 - val_loss: 0.0227
Epoch 9/200
394/394 - 1s - 2ms/step - accuracy: 0.9957 - loss: 0.0126 - val_accuracy: 0.9926 - val_loss

In [ ]:
test_results = classifier.evaluate(
    X_test,
    y_test_encoded,
    batch_size=BATCH_SIZE,
    verbose=0,
    return_dict=True,
)

In [ ]:
for metric_name, metric_value in test_results.items():
    print(
        f"{metric_name:12s}: "
        f"{metric_value:.6f}"
    )

accuracy    : 0.768497
loss        : 13.181582


In [ ]:
predicted_probabilities = classifier.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=0,
)

y_pred = np.argmax(
    predicted_probabilities,
    axis=1,
)

In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred,
)

weighted_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)

weighted_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)


confusion = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(NUMBER_OF_CLASSES),
)

class_support = confusion.sum(axis=1)
total_samples = class_support.sum()

per_class_fpr = []

for class_index in range(NUMBER_OF_CLASSES):
    tp = confusion[class_index, class_index]
    fn = confusion[class_index, :].sum() - tp
    fp = confusion[:, class_index].sum() - tp
    tn = total_samples - tp - fn - fp

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    per_class_fpr.append(fpr)

per_class_fpr = np.asarray(per_class_fpr)

weighted_fpr = np.average(
    per_class_fpr,
    weights=class_support,
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0,
)



print(f"Accuracy:           {accuracy:.6f}")
print(f"Weighted precision: {weighted_precision:.6f}")
print(f"Weighted recall:    {weighted_recall:.6f}")
print(f"False positive rate:{weighted_fpr:.6f}")
print(f"Weighted F1-score:  {weighted_f1:.6f}")
print("Confusion matrix")
print("----------------")
print(confusion)


Accuracy:           0.768497
Weighted precision: 0.785796
Weighted recall:    0.768497
False positive rate:0.151213
Weighted F1-score:  0.735697
Confusion matrix
----------------
[[8971  506  228    4    2]
 [1046 6311   66   35    0]
 [ 272  434 1684   23    8]
 [2327    3   85  336    3]
 [ 171    0    4    2   23]]


In [ ]:
classifier.save("/content/drive/MyDrive/Models_Table_5/DNN_Kunang.keras")
autoencoder.save("/content/drive/MyDrive/Models_Table_5/DAE_Kunang.keras")

# **Hyperparameter Optimization for 1D-CNN-Based Network Intrusion Detection Using GA and PSO [29]**

In [93]:
df = pd.concat([X_train_df, X_test_df])
df.describe(include="all")

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,142218.000000,142218.0,142218.0,142218.0,1.422180e+05,1.422180e+05,142218.000000,142218.000000,142218.000000,142218.000000,...,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000,142218.000000
unique,NaN,6.0,133.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,1.0,24.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,97540.0,38323.0,71205.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,277.798999,NaN,NaN,NaN,4.155920e+04,1.760007e+04,0.000218,0.020440,0.000211,0.187592,...,183.933440,119.613649,0.535200,0.084050,0.146106,0.030506,0.254900,0.250106,0.136882,0.137063
std,2462.514477,NaN,NaN,NaN,5.527968e+06,3.784560e+06,0.014762,0.239676,0.019842,1.997390,...,98.532002,111.258170,0.448051,0.194163,0.308942,0.108665,0.427749,0.428956,0.323371,0.335920
min,0.000000,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,87.000000,11.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,NaN,NaN,NaN,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,255.000000,72.000000,0.600000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,NaN,NaN,NaN,2.780000e+02,5.727500e+02,0.000000,0.000000,0.000000,0.000000,...,255.000000,255.000000,1.000000,0.070000,0.050000,0.010000,0.540000,0.470000,0.000000,0.000000


In [92]:
df['label'] = df['label'].map(lambda a: 0 if a == 'normal' else 1)
df.info()

KeyError: 'label'

In [ ]:
df.drop('num_outbound_cmds', axis=1, inplace=True)

In [ ]:
df.isnull().sum().sort_values(ascending = False)

,0
duration,0
protocol_type,0
service,0
flag,0
src_bytes,0
dst_bytes,0
land,0
wrong_fragment,0
urgent,0
hot,0


In [ ]:
df = df.drop_duplicates()

In [ ]:
df.describe(include="all")

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
count,147888.000000,147888,147888,147888,1.478880e+05,1.478880e+05,147888.000000,147888.000000,147888.000000,147888.000000,...,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000,147888.000000
unique,NaN,3,70,11,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,tcp,http,SF,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,121351,48168,89385,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,277.956508,NaN,NaN,NaN,4.039199e+04,1.716056e+04,0.000196,0.020434,0.000203,0.189982,...,119.403522,0.533860,0.084158,0.144168,0.030009,0.256827,0.251995,0.135880,0.136047,0.479559
std,2465.844177,NaN,NaN,NaN,5.421103e+06,3.711392e+06,0.014002,0.240056,0.019458,2.017317,...,111.204097,0.447948,0.193983,0.306437,0.106313,0.428872,0.430101,0.322299,0.334874,0.499584
min,0.000000,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,11.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,NaN,NaN,NaN,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,72.000000,0.590000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,NaN,NaN,NaN,2.780000e+02,5.810000e+02,0.000000,0.000000,0.000000,0.000000,...,255.000000,1.000000,0.070000,0.050000,0.010000,0.632500,0.550000,0.000000,0.000000,1.000000


In [ ]:
le_protocol_type = LabelEncoder()
le_service = LabelEncoder()
le_flag = LabelEncoder()

In [ ]:
df['protocol_type_encoded'] = le_protocol_type.fit_transform(df.protocol_type)
df['service_encoded'] = le_service.fit_transform(df.service)
df['flag_encoded'] = le_flag.fit_transform(df.flag)

/tmp/ipykernel_1808/4148780589.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['protocol_type_encoded'] = le_protocol_type.fit_transform(df.protocol_type)
/tmp/ipykernel_1808/4148780589.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['service_encoded'] = le_service.fit_transform(df.service)
/tmp/ipykernel_1808/4148780589.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in t

In [ ]:
protocol_type_ = OneHotEncoder()
service_ = OneHotEncoder()
flag_ = OneHotEncoder()


X = protocol_type_.fit_transform(df.protocol_type_encoded.values.reshape(-1,1)).toarray()
Xm = service_.fit_transform(df.service_encoded.values.reshape(-1,1)).toarray()
Xmm = flag_.fit_transform(df.flag_encoded.values.reshape(-1,1)).toarray()

dfOneHot = pd.DataFrame(X, columns=["protocol_type_"+str(int(i)) for i in range(X.shape[1])])
df = pd.concat([df.reset_index(drop=True), dfOneHot], axis=1)

dfOneHot2 = pd.DataFrame(Xm, columns=["service_"+str(int(i)) for i in range(Xm.shape[1])])
df = pd.concat([df.reset_index(drop=True), dfOneHot2], axis=1)

dfOneHot3 = pd.DataFrame(Xmm, columns=["flag_"+str(int(i)) for i in range(Xmm.shape[1])])
df = pd.concat([df.reset_index(drop=True), dfOneHot3], axis=1)

In [ ]:
df.drop(columns=['protocol_type','service','flag'], inplace = True )
df.shape

(147888, 125)

In [ ]:
bestfeatuers=SelectKBest(score_func=chi2,k=100)
inp=df.iloc[:,0:125]
fit=bestfeatuers.fit(inp,df['label'])
dfscores=pd.DataFrame(fit.scores_)
dfcol=pd.DataFrame(inp.columns)
featurescore=pd.concat([dfcol,dfscores],axis=1)
featurescore.columns=['feature','score']
k=featurescore['feature']
t=featurescore.nlargest(100,'score')
li=list(t['feature'])
print("Top 100 features:")
#print("\n")
for x in li:
    print(x,end=", ")

X=df[li].drop(columns=['label'])
y=df[li]['label']

Top 100 features:
src_bytes, dst_bytes, duration, dst_host_srv_count, count, dst_host_count, service_encoded, label, flag_encoded, logged_in, dst_host_srv_serror_rate, flag_5, srv_serror_rate, serror_rate, dst_host_serror_rate, num_root, service_24, flag_9, num_compromised, dst_host_same_srv_rate, service_49, same_srv_rate, rerror_rate, srv_rerror_rate, dst_host_srv_rerror_rate, dst_host_rerror_rate, srv_count, service_12, flag_1, protocol_type_2, service_54, protocol_type_0, dst_host_diff_srv_rate, diff_srv_rate, service_14, service_15, wrong_fragment, flag_4, protocol_type_encoded, flag_2, srv_diff_host_rate, service_60, service_28, num_file_creations, service_2, service_66, service_20, service_6, service_5, service_29, service_67, service_69, service_41, service_68, service_47, service_8, service_7, service_58, service_26, service_10, service_21, service_9, service_57, service_16, dst_host_same_src_port_rate, service_33, service_59, service_65, service_17, service_36, service_11, se

In [ ]:
# Split the data into train (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Split the training data further into train (80%) and validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=42)

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

input_shape = (X_train.shape[1], 1)
output_classes = 1

In [ ]:
model = Sequential()

model.add(Conv1D(filters=32, kernel_size=9, activation='relu', input_shape=(input_shape)))
model.add(MaxPooling1D(pool_size=5))

model.add(Conv1D(filters=64, kernel_size=9, activation='relu'))
model.add(MaxPooling1D(pool_size=5))

model.add(Dropout(0.253378))
model.add(Flatten())

for _ in range(1):
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.253378))

model.add(Dense(output_classes, activation='sigmoid'))

optimizer = Adam(learning_rate=0.000471)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 91, 32)         │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 18, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 10, 64)         │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 2, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,457 (138.50 KB)

 Trainable params: 35,457 (138.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train, y_train, batch_size=32, epochs=72, validation_data=(X_val, y_val))

Epoch 1/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.9516 - loss: 0.1259 - val_accuracy: 0.9720 - val_loss: 0.0712
Epoch 2/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9727 - loss: 0.0730 - val_accuracy: 0.9784 - val_loss: 0.0557
Epoch 3/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9780 - loss: 0.0590 - val_accuracy: 0.9808 - val_loss: 0.0466
Epoch 4/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9803 - loss: 0.0532 - val_accuracy: 0.9822 - val_loss: 0.0441
Epoch 5/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9815 - loss: 0.0494 - val_accuracy: 0.9830 - val_loss: 0.0428
Epoch 6/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9825 - loss: 0.0467 - val_accuracy: 0.9830 - val_loss: 0.0444
Epoch 7/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9833 - loss: 0.0444 - val_accuracy: 0.9855 - val_loss: 0.0374
Epoch 8/72
3236/3236 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9840 - loss: 0.0430 -

In [ ]:
y_pred = model.predict(np.expand_dims(X_test, axis=2))
y_pred = (y_pred > 0.5)


cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")


925/925 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Confusion matrix
----------------
[[15145   127]
 [  138 14168]]

Attack-class metrics
--------------------
Accuracy : 0.9910 (99.10%)
Detection rate : 0.9904 (99.04%)
Precision      : 0.9911 (99.11%)
FPR            : 0.0083 (0.83%)
F1 score       : 0.9907 (99.07%)


In [ ]:
model.save("/content/drive/MyDrive/Models_Table_5/1DCNN_Kilichev.keras")

# **Towards Detection of Network Anomalies using Machine Learning Algorithms on the NSL-KDD Benchmark Datasets [68]**

In [97]:
RANDOM_STATE = 100

X_train_df, X_validation_df, y_train, y_validation = (
    train_test_split(
        X_train_all_df,
        y_train_all,
        test_size=0.30,
        random_state=RANDOM_STATE,
        shuffle=True,
    )
)

In [98]:
categorical_columns = [
    "protocol_type",
    "service",
    "flag",
]

numerical_columns = [
    column
    for column in X_train_df.columns
    if column not in categorical_columns
]

In [99]:
one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

min_max_scaler = MinMaxScaler(
    feature_range=(0.0, 1.0),
    clip=True,
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            min_max_scaler,
            numerical_columns,
        ),
        (
            "categorical",
            one_hot_encoder,
            categorical_columns,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)


In [100]:
X_train_processed = preprocessor.fit_transform(
    X_train_df
).astype(np.float32)

X_validation_processed = preprocessor.transform(
    X_validation_df
).astype(np.float32)

X_test_processed = preprocessor.transform(
    X_test_df
).astype(np.float32)


In [101]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=100,
    n_jobs=-1,
)

random_forest.fit(
    X_train_processed,
    y_train,
)

RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=100)

In [102]:

encoded_feature_names = (
    preprocessor.get_feature_names_out()
)

feature_importances = (
    random_forest.feature_importances_
)

importance_threshold = 0.037

selected_feature_mask = (
    feature_importances >= importance_threshold
)

selected_feature_indices = np.where(
    selected_feature_mask
)[0]

selected_feature_names = encoded_feature_names[
    selected_feature_mask
]

print("Importance threshold:", importance_threshold)
print(
    "Number of selected features:",
    len(selected_feature_names),
)

Importance threshold: 0.037
Number of selected features: 9


In [103]:
feature_importance_df = pd.DataFrame(
    {
        "feature": encoded_feature_names,
        "importance": feature_importances,
        "selected": selected_feature_mask,
    }
)

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(feature_importance_df.head(20))

                                   feature  importance  selected
0                     numerical__src_bytes    0.132624      True
1                     numerical__dst_bytes    0.101803      True
2                     categorical__flag_SF    0.063152      True
3            numerical__dst_host_srv_count    0.050663      True
4        numerical__dst_host_same_srv_rate    0.049871      True
5                 numerical__diff_srv_rate    0.049460      True
6                 numerical__same_srv_rate    0.044356      True
7        numerical__dst_host_diff_srv_rate    0.041900      True
8                         numerical__count    0.040663      True
9                     numerical__logged_in    0.033835     False
10     numerical__dst_host_srv_serror_rate    0.028551     False
11  numerical__dst_host_same_src_port_rate    0.025404     False
12         numerical__dst_host_rerror_rate    0.023471     False
13               numerical__dst_host_count    0.022565     False
14         numerical__dst

In [104]:
X_train_selected = X_train_processed[
    :,
    selected_feature_mask,
]

X_validation_selected = X_validation_processed[
    :,
    selected_feature_mask,
]

X_test_selected = X_test_processed[
    :,
    selected_feature_mask,
]

print("Selected training shape:  ", X_train_selected.shape)
print("Selected validation shape:", X_validation_selected.shape)
print("Selected testing shape:   ", X_test_selected.shape)

Selected training shape:   (103961, 9)
Selected validation shape: (44556, 9)
Selected testing shape:    (22544, 9)


In [105]:
knn_classifier = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    algorithm="auto",
    leaf_size=30,
    metric="euclidean",
    n_jobs=-1,
)

In [106]:
knn_classifier.fit(
    X_train_selected,
    y_train,
)

KNeighborsClassifier(metric='euclidean', n_jobs=-1)

In [107]:
_ = knn_classifier.predict_proba(
    X_validation_selected[:100]
)

In [108]:
test_probabilities = (
    knn_classifier.predict_proba(
        X_test_selected
    )
)

test_anomaly_probabilities = (
    test_probabilities[:, 1]
)

test_predictions = (
    test_anomaly_probabilities >= 0.5
).astype(np.int32)


test_confusion = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1],
)

accuracy = accuracy_score(
    y_test,
    test_predictions,
)

precision = precision_score(
    y_test,
    test_predictions,
    pos_label=1,
    zero_division=0,
)

recall = recall_score(
    y_test,
    test_predictions,
    pos_label=1,
    zero_division=0,
)

test_tn = test_confusion[0, 0]
test_fp = test_confusion[0, 1]
test_fn = test_confusion[1, 0]
test_tp = test_confusion[1, 1]

false_positive_rate = test_fp / (
    test_fp + test_tn
)

f1 = f1_score(
    y_test,
    test_predictions,
    pos_label=1,
    zero_division=0,
)

In [109]:
print("Confusion matrix")
print("----------------")
print(test_confusion)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[ 9213   498]
 [ 1397 11436]]

Attack-class metrics
--------------------
Accuracy : 0.9159 (91.59%)
Detection rate : 0.8911 (89.11%)
Precision      : 0.9583 (95.83%)
FPR            : 0.0513 (5.13%)
F1 score       : 0.9235 (92.35%)


In [ ]:
import joblib

saved_bundle = {
    "preprocessor": preprocessor,
    "random_forest": random_forest,
    "selected_feature_mask": selected_feature_mask,
    "encoded_feature_names": encoded_feature_names,
    "selected_feature_names": selected_feature_names,
    "knn_classifier": knn_classifier,
    "categorical_columns": categorical_columns,
    "numerical_columns": numerical_columns,
    "original_feature_columns": list(
        X_train_all_df.columns
    ),
    "class_names": ["Normal", "Anomaly"],
}

joblib.dump(
    saved_bundle,
    "/content/drive/MyDrive/Models_Table_7/KNN_Vibhute.joblib",
)

['/content/drive/MyDrive/Models_Table_7/KNN_Vibhute.joblib']

# **Ckan: Convolutional kolmogorov–arnold networks model for intrusion detection in iot environment [3]**

In [110]:
import random
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

from torch.utils.data import DataLoader, TensorDataset

In [121]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 20
VALIDATION_SIZE = 0.10

NUMBER_OF_CLASSES = 2

CLASS_NAMES = [
    "Normal",
    "Attack",
]

Device: cuda


In [113]:
train_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Train.csv",
    names=column_names,
    header=None,
)

test_df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/DATASETS/NSLKDD/NSL_KDD_Test.csv",
    names=column_names,
    header=None,
)

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

print(train_df.head())

Training data shape: (125973, 43)
Testing data shape: (22544, 43)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1     

In [114]:
train_labels = (
    all_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.lower()
)

test_labels = (
    test_df["label"]
    .astype(str)
    .str.strip()
    .str.rstrip(".")
    .str.lower()
)




y_train_all = (
    train_labels != "normal"
).astype(np.int64).to_numpy()

y_test = (
    test_labels != "normal"
).astype(np.int64).to_numpy()


print("Training label values:", np.unique(y_train_all))
print("Testing label values:", np.unique(y_test))

Training label values: [0 1]
Testing label values: [0 1]


In [115]:
X_train_all_df = all_df.drop(
    columns=[
        "label",
        "difficulty",
    ]
).copy()

X_test_df = test_df.drop(
    columns=[
        "label",
        "difficulty",
    ]
).copy()


print("Raw training feature shape:", X_train_all_df.shape)
print("Raw testing feature shape:", X_test_df.shape)

Raw training feature shape: (148517, 41)
Raw testing feature shape: (22544, 41)


In [122]:
(
    X_train_df,
    X_validation_df,
    y_train,
    y_validation,
) = train_test_split(
    X_train_all_df,
    y_train_all,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=y_train_all,
    shuffle=True,
)


print("Internal training shape:", X_train_df.shape)
print("Validation shape:       ", X_validation_df.shape)
print("Official test shape:    ", X_test_df.shape)

Internal training shape: (133665, 41)
Validation shape:        (14852, 41)
Official test shape:     (22544, 41)


In [123]:
categorical_columns = [
    "protocol_type",
    "service",
    "flag",
]

numerical_columns = [
    column
    for column in X_train_df.columns
    if column not in categorical_columns
]


print("Categorical columns:", categorical_columns)
print("Number of numerical columns:", len(numerical_columns))

Categorical columns: ['protocol_type', 'service', 'flag']
Number of numerical columns: 38


In [124]:
one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32,
)


min_max_scaler = MinMaxScaler(
    feature_range=(0.0, 1.0),
    clip=True,
)

preprocessor = ColumnTransformer(
transformers=[
    (
        "numerical",
        min_max_scaler,
        numerical_columns,
    ),
    (
        "categorical",
        one_hot_encoder,
        categorical_columns,
    ),
],
remainder="drop",
sparse_threshold=0.0,
)

In [125]:
X_train_processed = preprocessor.fit_transform(
    X_train_df
).astype(np.float32)

X_validation_processed = preprocessor.transform(
    X_validation_df
).astype(np.float32)

X_test_processed = preprocessor.transform(
    X_test_df
).astype(np.float32)

In [126]:
print("Processed training shape:  ", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed testing shape:   ", X_test_processed.shape)

Processed training shape:   (133665, 122)
Processed validation shape: (14852, 122)
Processed testing shape:    (22544, 122)


In [127]:
X_train_tensor = torch.from_numpy(
    X_train_processed
).unsqueeze(1)

X_validation_tensor = torch.from_numpy(
    X_validation_processed
).unsqueeze(1)

X_test_tensor = torch.from_numpy(
    X_test_processed
).unsqueeze(1)


y_train_tensor = torch.from_numpy(
    y_train
).long()

y_validation_tensor = torch.from_numpy(
    y_validation
).long()

y_test_tensor = torch.from_numpy(
    y_test
).long()


print("Training tensor shape:", X_train_tensor.shape)
print("Validation tensor shape:", X_validation_tensor.shape)
print("Testing tensor shape:", X_test_tensor.shape)

Training tensor shape: torch.Size([133665, 1, 122])
Validation tensor shape: torch.Size([14852, 1, 122])
Testing tensor shape: torch.Size([22544, 1, 122])


In [128]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

validation_dataset = TensorDataset(
    X_validation_tensor,
    y_validation_tensor,
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor,
)


pin_memory = DEVICE.type == "cuda"


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

In [129]:
class KANLinear(nn.Module):

    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        grid_range=(-1.0, 1.0),
    ):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.number_of_spline_coefficients = (
            grid_size + spline_order
        )

        grid_step = (
            grid_range[1] - grid_range[0]
        ) / grid_size

        grid = (
            torch.arange(
                -spline_order,
                grid_size + spline_order + 1,
                dtype=torch.float32,
            )
            * grid_step
            + grid_range[0]
        )

        grid = grid.expand(
            in_features,
            -1,
        ).contiguous()

        self.register_buffer(
            "grid",
            grid,
        )

        # One base weight per connection
        self.base_weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )

        # grid_size + spline_order = 8 coefficients
        self.spline_weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
                self.number_of_spline_coefficients,
            )
        )

        # One spline scale per connection
        self.spline_scaler = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
            )
        )

        nn.init.kaiming_uniform_(
            self.base_weight,
            a=np.sqrt(5),
        )

        nn.init.normal_(
            self.spline_weight,
            mean=0.0,
            std=0.02,
        )

        nn.init.ones_(
            self.spline_scaler
        )


    def b_splines(self, x):

        # x shape:
        # (batch_size, in_features)

        x_expanded = x.unsqueeze(-1)

        grid = self.grid

        bases = (
            (
                x_expanded >= grid[:, :-1]
            )
            &
            (
                x_expanded < grid[:, 1:]
            )
        ).to(x.dtype)

        for order in range(
            1,
            self.spline_order + 1,
        ):
            left_numerator = (
                x_expanded
                - grid[:, :-(order + 1)]
            )

            left_denominator = (
                grid[:, order:-1]
                - grid[:, :-(order + 1)]
            )

            right_numerator = (
                grid[:, order + 1:]
                - x_expanded
            )

            right_denominator = (
                grid[:, order + 1:]
                - grid[:, 1:-order]
            )

            left_term = (
                left_numerator
                / left_denominator
                * bases[:, :, :-1]
            )

            right_term = (
                right_numerator
                / right_denominator
                * bases[:, :, 1:]
            )

            bases = left_term + right_term

        return bases.contiguous()


    def forward(self, x):

        # Base branch
        base_output = F.linear(
            F.silu(x),
            self.base_weight,
        )

        # B-spline branch
        spline_bases = self.b_splines(x)

        scaled_spline_weights = (
            self.spline_weight
            * self.spline_scaler.unsqueeze(-1)
        )

        spline_output = F.linear(
            spline_bases.reshape(
                x.shape[0],
                -1,
            ),
            scaled_spline_weights.reshape(
                self.out_features,
                -1,
            ),
        )

        return base_output + spline_output

In [130]:
class CKANBinary(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=0,
            bias=True,
        )

        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=0,
            bias=True,
        )

        self.adaptive_pool = (
            nn.AdaptiveAvgPool1d(
                output_size=1
            )
        )

        self.kan = KANLinear(
            in_features=64,
            out_features=2,
            grid_size=5,
            spline_order=3,
            grid_range=(-1.0, 1.0),
        )


    def forward(self, x):

        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.adaptive_pool(x)

        # (batch, 64, 1) -> (batch, 64)
        x = x.squeeze(-1)

        logits = self.kan(x)

        return logits

In [131]:
model = CKANBinary().to(DEVICE)

print(model)

total_trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    "Total trainable parameters:",
    total_trainable_parameters,
)

CKANBinary(
  (conv1): Conv1d(1, 32, kernel_size=(3,), stride=(1,))
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,))
  (adaptive_pool): AdaptiveAvgPool1d(output_size=1)
  (kan): KANLinear()
)
Total trainable parameters: 7616


In [132]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

In [133]:
training_loss_history = []
validation_loss_history = []

training_accuracy_history = []
validation_accuracy_history = []


if DEVICE.type == "cuda":
    torch.cuda.synchronize()

training_start_time = time.perf_counter()


for epoch in range(EPOCHS):


    model.train()

    training_loss_sum = 0.0
    training_correct = 0
    training_samples = 0


    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(
            DEVICE,
            non_blocking=True,
        )

        batch_labels = batch_labels.to(
            DEVICE,
            non_blocking=True,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_labels,
        )


        loss.backward()

        optimizer.step()


        batch_size = batch_features.size(0)

        training_loss_sum += (
            loss.item() * batch_size
        )

        training_predictions = torch.argmax(
            logits,
            dim=1,
        )

        training_correct += (
            training_predictions
            == batch_labels
        ).sum().item()

        training_samples += batch_size


    epoch_training_loss = (
        training_loss_sum
        / training_samples
    )

    epoch_training_accuracy = (
        training_correct
        / training_samples
    )


    model.eval()

    validation_loss_sum = 0.0
    validation_correct = 0
    validation_samples = 0


    with torch.no_grad():

        for (
            batch_features,
            batch_labels,
        ) in validation_loader:

            batch_features = batch_features.to(
                DEVICE,
                non_blocking=True,
            )

            batch_labels = batch_labels.to(
                DEVICE,
                non_blocking=True,
            )


            logits = model(
                batch_features
            )

            loss = criterion(
                logits,
                batch_labels,
            )


            batch_size = batch_features.size(0)

            validation_loss_sum += (
                loss.item() * batch_size
            )

            validation_predictions = torch.argmax(
                logits,
                dim=1,
            )

            validation_correct += (
                validation_predictions
                == batch_labels
            ).sum().item()

            validation_samples += batch_size


    epoch_validation_loss = (
        validation_loss_sum
        / validation_samples
    )

    epoch_validation_accuracy = (
        validation_correct
        / validation_samples
    )


    training_loss_history.append(
        epoch_training_loss
    )

    validation_loss_history.append(
        epoch_validation_loss
    )

    training_accuracy_history.append(
        epoch_training_accuracy
    )

    validation_accuracy_history.append(
        epoch_validation_accuracy
    )


    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train loss: {epoch_training_loss:.6f} | "
        f"train acc: {epoch_training_accuracy:.4f} | "
        f"val loss: {epoch_validation_loss:.6f} | "
        f"val acc: {epoch_validation_accuracy:.4f}"
    )


if DEVICE.type == "cuda":
    torch.cuda.synchronize()

training_time = (
    time.perf_counter()
    - training_start_time
)


print(
    f"\nTotal training time: "
    f"{training_time:.3f} seconds"
)

Epoch 01/20 | train loss: 0.356947 | train acc: 0.8610 | val loss: 0.290460 | val acc: 0.8903
Epoch 02/20 | train loss: 0.247209 | train acc: 0.8982 | val loss: 0.223625 | val acc: 0.9053
Epoch 03/20 | train loss: 0.215121 | train acc: 0.9100 | val loss: 0.199446 | val acc: 0.9115
Epoch 04/20 | train loss: 0.196087 | train acc: 0.9200 | val loss: 0.186112 | val acc: 0.9278
Epoch 05/20 | train loss: 0.184606 | train acc: 0.9267 | val loss: 0.173269 | val acc: 0.9342
Epoch 06/20 | train loss: 0.176511 | train acc: 0.9316 | val loss: 0.169532 | val acc: 0.9334
Epoch 07/20 | train loss: 0.167702 | train acc: 0.9365 | val loss: 0.159841 | val acc: 0.9416
Epoch 08/20 | train loss: 0.162698 | train acc: 0.9385 | val loss: 0.151799 | val acc: 0.9445
Epoch 09/20 | train loss: 0.158181 | train acc: 0.9404 | val loss: 0.153503 | val acc: 0.9413
Epoch 10/20 | train loss: 0.153799 | train acc: 0.9423 | val loss: 0.149163 | val acc: 0.9406
Epoch 11/20 | train loss: 0.150604 | train acc: 0.9438 | val

In [134]:
model.eval()

test_loss_sum = 0.0
test_sample_count = 0

all_test_labels = []
all_test_predictions = []
all_test_attack_probabilities = []


with torch.no_grad():

    for (
        batch_features,
        batch_labels,
    ) in test_loader:

        batch_features = batch_features.to(
            DEVICE,
            non_blocking=True,
        )

        batch_labels = batch_labels.to(
            DEVICE,
            non_blocking=True,
        )


        logits = model(
            batch_features
        )

        loss = criterion(
            logits,
            batch_labels,
        )


        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = torch.argmax(
            logits,
            dim=1,
        )


        batch_size = batch_features.size(0)

        test_loss_sum += (
            loss.item() * batch_size
        )

        test_sample_count += batch_size


        all_test_labels.append(
            batch_labels.cpu().numpy()
        )

        all_test_predictions.append(
            predictions.cpu().numpy()
        )

        all_test_attack_probabilities.append(
            probabilities[:, 1]
            .cpu()
            .numpy()
        )


y_test_true = np.concatenate(
    all_test_labels
)

y_test_pred = np.concatenate(
    all_test_predictions
)

y_test_attack_probability = np.concatenate(
    all_test_attack_probabilities
)


test_loss = (
    test_loss_sum
    / test_sample_count
)

In [135]:

accuracy = accuracy_score(
    y_test_true,
    y_test_pred,
)

precision = precision_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0,
)

recall = recall_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0,
)

f1 = f1_score(
    y_test_true,
    y_test_pred,
    pos_label=1,
    zero_division=0,
)

cm = confusion_matrix(
    y_test_true,
    y_test_pred,
    labels=[0, 1],
)


true_negative = cm[0, 0]
false_positive = cm[0, 1]
false_negative = cm[1, 0]
true_positive = cm[1, 1]

fpr = (false_positive/ (false_positive+ true_negative))


In [136]:
print("Confusion matrix")
print("----------------")
print(confusion_matrix)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {fpr:.4f} "
      f"({fpr * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
<function confusion_matrix at 0x7e0f21137a60>

Attack-class metrics
--------------------
Accuracy : 0.8865 (88.65%)
Detection rate : 0.8298 (82.98%)
Precision      : 0.9661 (96.61%)
FPR            : 0.0385 (3.85%)
F1 score       : 0.8928 (89.28%)
